# GraphDataset / SubgraphDataset amb dades bibliogràfiques (OpenAlex)

Aquest notebook mostra el `torch_dataloader` de `cvcdocdb` sobre el dataset
bibliogràfic d'exemple (`cvcdocdb.exemples.load_bibliografia_openalex`):

- `GraphDataset` + `GraphDataLoader`: streaming pla de nodes (`Paper`, `Author`).
- `SubgraphDataset` + `PyGDataLoader`: subgrafs ego k-hop com a objectes
  `torch_geometric.data.Data`, amb un embedding per `Paper` (`data.x` i `data.emb`).
- `MetaPath2Vec`: entrenament ràpid d'embeddings sobre el graf heterogeni
  `Author`–`Paper` seguint el metapath `author -writes-> paper -cites-> paper -written_by-> author`.

Com que el dataset d'OpenAlex no porta embeddings reals, es genera primer un
embedding determinista (hash del títol) només a efectes de demostració — i
més endavant s'entrena un embedding real amb MetaPath2Vec, que pots fer
servir per substituir-lo.

In [ ]:
import importlib.util

# Comprovació de presència del paquet
package_to_check = 'cvcdocdb'
spec = importlib.util.find_spec(package_to_check)

if spec is None:
    print(f'⚠️ {package_to_check} no està instal·lat. Iniciant instal·lació...')
    %pip install -q --upgrade cvcdocdb
    print("✅ Instal·lació completada. L'estat del kernel PODRIA requerir un reinici.")
else:
    print(f'✅ {package_to_check} ja està present al sistema. Saltant instal·lació.')


In [ ]:
import importlib.util

for package_to_check in ('torch', 'torch_geometric'):
    spec = importlib.util.find_spec(package_to_check)
    if spec is None:
        print(f'⚠️ {package_to_check} no està instal·lat. Iniciant instal·lació...')
        %pip install -q {package_to_check}
        print("✅ Instal·lació completada. L'estat del kernel PODRIA requerir un reinici.")
    else:
        print(f'✅ {package_to_check} ja està present al sistema. Saltant instal·lació.')


## 1. Carregar el dataset bibliogràfic a `NetworkXGraph`

In [ ]:
from cvcdocdb import NetworkXGraph
from cvcdocdb.exemples import load_bibliografia_openalex

store = NetworkXGraph()
stats = load_bibliografia_openalex(store, query="graph database", per_page=15)
print("Dataset stats:", stats)


## 2. Afegir un embedding de demostració a cada `Paper`

`load_bibliografia_openalex` no genera embeddings. Aquí en calculem un de
determinista (hash del títol → vector normalitzat) i l'afegim com a atribut
`embedding` de cada node `Paper` amb `insertNode(..., update=True)`, que fa
merge de l'atribut sense tocar la resta del node.

In [ ]:
import hashlib

from cvcdocdb.base import Node

EMB_DIM = 16


def toy_embedding(text: str, dim: int = EMB_DIM) -> list[float]:
    """Embedding determinista (només per a demo) a partir del hash del text."""
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    raw = [digest[i % len(digest)] / 255.0 for i in range(dim)]
    norm = sum(v * v for v in raw) ** 0.5 or 1.0
    return [v / norm for v in raw]


paper_ids = store.find_nodes_by_property("main_label", "Paper")
for nid in paper_ids:
    attrs = store.get_node_attrs(nid)
    title = attrs.get("title", "")
    openalex_id = attrs.get("openalex_id")
    embedding = toy_embedding(title)
    store.insertNode(
        Node(pk={"openalex_id": openalex_id}, main_label="Paper", embedding=embedding),
        update=True,
    )

print(f"Embeddings afegits a {len(paper_ids)} papers (dim={EMB_DIM}).")


## 3. Streaming pla amb `GraphDataset` / `GraphDataLoader`

In [ ]:
from cvcdocdb.torch_dataloader import GraphDataset, GraphDataLoader

flat_ds = GraphDataset(store, label_filter="Paper")
flat_loader = GraphDataLoader(flat_ds, batch_size=8, num_workers=0)

batch = next(iter(flat_loader))
print("node_id:", batch["node_id"])
print("main_label:", batch["main_label"][:3], "...")


## 4. Subgrafs ego amb embeddings via `SubgraphDataset` / `PyGDataLoader`

Cada `Paper` és una seed; el subgraf k-hop inclou els seus coautors i papers
citats/citants. `vector_attr="embedding"` fusiona l'embedding a `data.x` i el
deixa també accessible per separat a `data.emb`.

In [ ]:
from cvcdocdb.torch_dataloader import SubgraphDataset, PyGDataLoader

sub_ds = SubgraphDataset(
    store,
    hops=1,
    node_attrs=["year"],
    vector_attr="embedding",
    label_filter="Paper",
    missing=0.0,
)
sub_loader = PyGDataLoader(sub_ds, batch_size=4)

pyg_batch = next(iter(sub_loader))
print("x:", pyg_batch.x.shape)          # [N, 1 (year) + 16 (embedding)]
print("emb:", pyg_batch.emb.shape)      # [N, 16]
print("edge_index:", pyg_batch.edge_index.shape)
print("node_ids:", pyg_batch.node_ids[:10])


## 5. Entrenament d'embeddings amb `MetaPath2Vec`

En comptes de l'embedding "de joguina" de la secció 2, entrenem un embedding
real amb `torch_geometric.nn.MetaPath2Vec` sobre el graf heterogeni complet
`Author`/`Paper`, seguint el metapath:

```
Author --AUTHORED--> Paper --CITES--> Paper --AUTHORED_BY--> Author
```

`to_hetero_edge_index_dict` carrega tot el graf en una sola passada pels
edges (sense atributs de node, el més lleuger possible) i retorna
l'`edge_index_dict`, el nombre de nodes per tipus i el mapeig id original
→ índex local que necessita `MetaPath2Vec`.

Com que el dataset és petit (una mostra d'OpenAlex de ~15 papers), l'entrenament
és breu (poques èpoques) i només a efectes il·lustratius — en un dataset real
cal augmentar `walk_length`, `walks_per_node`, `context_size` i el nombre
d'èpoques.

In [ ]:
import torch
from torch_geometric.nn import MetaPath2Vec

from cvcdocdb.torch_dataloader import to_hetero_edge_index_dict

edge_index_dict, num_nodes_dict, node_maps = to_hetero_edge_index_dict(store)
print("Tipus de node:", num_nodes_dict)
print("Tipus de relació:", list(edge_index_dict))

# Afegim la relació inversa Paper->Author perquè el metapath pugui tancar el cicle.
authored = edge_index_dict[("Author", "AUTHORED", "Paper")]
edge_index_dict[("Paper", "AUTHORED_BY", "Author")] = authored.flip(0)

if ("Paper", "CITES", "Paper") not in edge_index_dict:
    # Mostra petita: pot no haver-hi cap citació intra-mostra. Afegim
    # self-loops perquè el pas paper->paper del metapath sigui transitable.
    n_papers = num_nodes_dict["Paper"]
    idx = torch.arange(n_papers, dtype=torch.long)
    edge_index_dict[("Paper", "CITES", "Paper")] = torch.stack([idx, idx])

metapath = [
    ("Author", "AUTHORED", "Paper"),
    ("Paper", "CITES", "Paper"),
    ("Paper", "AUTHORED_BY", "Author"),
]

model = MetaPath2Vec(
    edge_index_dict,
    embedding_dim=16,
    metapath=metapath,
    walk_length=4,
    context_size=2,
    walks_per_node=3,
    num_negative_samples=3,
    num_nodes_dict=num_nodes_dict,
    sparse=True,
)

loader = model.loader(batch_size=8, shuffle=True)
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

model.train()
for epoch in range(5):
    total_loss = 0.0
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = model.loss(pos_rw, neg_rw)
        loss.backward()
        optimizer.step()
        total_loss += float(loss)
    print(f"epoch {epoch + 1}: loss={total_loss / max(len(loader), 1):.4f}")

paper_embeddings = model("Paper").detach()
print("Paper embeddings (MetaPath2Vec):", paper_embeddings.shape)


Amb l'embedding après podem, per exemple, mirar quins papers de la mostra
són més similars entre ells (cosinus) — un ús típic d'aquests embeddings és
alimentar-los de nou al graf (`insertNode(..., update=True)`) per fer-los
servir amb `SubgraphDataset(vector_attr=...)` com a la secció 4.

In [ ]:
import torch.nn.functional as F

sample = paper_embeddings[: min(5, len(paper_ids))]
sim = F.cosine_similarity(sample.unsqueeze(1), sample.unsqueeze(0), dim=-1)
print("Similitud cosinus entre els primers papers:")
print(sim)


## 6. Link prediction: predir citacions entre papers

Amb els embeddings de `Paper` ja entrenats, fem un exemple senzill de
**link prediction**: predir si existeix una relació `CITES` entre dos
papers. Com que `MetaPath2Vec` només aprèn embeddings *de node*, calem
la representació d'una aresta combinant els dos extrems amb
`edge_embeddings(..., op="dot")` — el producte escalar és directament un
score de similitud/existència.

Partim les arestes `CITES` per any (`split_edges_by_node_property`):
entrenem amb parells de papers "antics" i validem amb parells "recents".
Com que necessitem també exemples negatius (parells sense citació),
usem `torch_geometric.utils.negative_sampling`.

In [ ]:
import torch
from torch_geometric.utils import negative_sampling

from cvcdocdb.torch_dataloader import edge_embeddings, split_edges_by_node_property

cites_edge_index = edge_index_dict[("Paper", "CITES", "Paper")]

# Any de publicació per paper, alineat amb node_maps["Paper"] (índex local).
local_to_orig_paper = {v: k for k, v in node_maps["Paper"].items()}
years = torch.tensor([
    float((store.get_node_attrs(local_to_orig_paper[i]) or {}).get("year") or 0)
    for i in range(num_nodes_dict["Paper"])
])

cutoff = years.median()
splits = split_edges_by_node_property(
    cites_edge_index, years,
    train=lambda s, d: torch.maximum(s, d) <= cutoff,
    test=lambda s, d: torch.minimum(s, d) >= cutoff,
)
print("Cutoff (any mitjà):", cutoff.item())
print("Arestes CITES — train:", splits["train"].shape[1], " test:", splits["test"].shape[1])


In [ ]:
n_papers = num_nodes_dict["Paper"]


def score_split(pos_edge_index, name):
    if pos_edge_index.shape[1] == 0:
        print(f"{name}: cap aresta positiva, salto.")
        return
    neg_edge_index = negative_sampling(
        edge_index=cites_edge_index,
        num_nodes=n_papers,
        num_neg_samples=pos_edge_index.shape[1],
    )
    pos_scores = edge_embeddings(paper_embeddings, pos_edge_index, op="dot")
    neg_scores = edge_embeddings(paper_embeddings, neg_edge_index, op="dot")

    preds = torch.cat([pos_scores, neg_scores])
    labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
    accuracy = ((preds > 0).float() == labels).float().mean()

    print(f"{name}: score mitjà positiu={pos_scores.mean():.3f}  "
          f"negatiu={neg_scores.mean():.3f}  accuracy(llindar=0)={accuracy:.2%}")


score_split(splits["train"], "train")
score_split(splits["test"], "test")


In [ ]:
store.close()